In [1]:
import pathlib
from pathlib import Path
import sys
import pyarrow.parquet as pq
from DataFile.data_name import DataName
from typing import List

In [2]:
import dask.dataframe as dd
from pathlib import Path

# 获取 Parquet 文件列表
pfs = list(Path(r"E:\tmp\OKX-Books-BTC-USDT-400").glob("*.parquet"))

# 使用 Dask 读取多个 Parquet 文件
ddf = dd.read_parquet(pfs, columns=["ts"])  # 只读取需要的列 'ts'

# 将每个分区与文件路径配对，并转换为字典
tss = {}
for i, pf in enumerate(pfs):
    # 获取对应分区的 ts 列并计算
    partition = ddf.partitions[i]
    tss[pf] = partition["ts"].compute().to_list()

In [3]:
from bisect import bisect_left, bisect_right
def count_overlapping_items(ts_list1: List[int], ts_list2: List[int], is_sorted: bool) -> int:
    """
    计算两个时间戳列表在重叠时间段内的相同元素数量。
    
    Args:
        ts_list1 (List[int]): 第一个文件的时间戳列表
        ts_list2 (List[int]): 第二个文件的时间戳列表
    
    Returns:
        int: 重叠时间段内相同的时间戳数量
    """
    if not ts_list1 or not ts_list2:
        return 0
    
    if not is_sorted:
        ts_list1 = sorted(ts_list1)
        ts_list2 = sorted(ts_list2)

    def fix_one(ts_list1, ts_list2):
        start = bisect_left(ts_list1, ts_list2[0])
        end = bisect_right(ts_list1, ts_list2[-1])
        return end - start

    return max(fix_one(ts_list1, ts_list2), fix_one(ts_list2, ts_list1))

In [4]:
for i in range(len(pfs)-1):
    ts1 = tss[pfs[i]]
    ts2 = tss[pfs[i+1]]
    
    print(f"{pfs[i].name} vs {pfs[i+1].name}: {count_overlapping_items(ts1, ts2, True)} items overlap")

OKX-Books-BTC-USDT-400-1743436790407-1743537950600.parquet vs OKX-Books-BTC-USDT-400-1743537950700-1743671559505.parquet: 0 items overlap
OKX-Books-BTC-USDT-400-1743537950700-1743671559505.parquet vs OKX-Books-BTC-USDT-400-1743671559605-1743772112206.parquet: 0 items overlap
OKX-Books-BTC-USDT-400-1743671559605-1743772112206.parquet vs OKX-Books-BTC-USDT-400-1743772112306-1743908302903.parquet: 0 items overlap
OKX-Books-BTC-USDT-400-1743772112306-1743908302903.parquet vs OKX-Books-BTC-USDT-400-1743908303003-1744009469306.parquet: 0 items overlap
OKX-Books-BTC-USDT-400-1743908303003-1744009469306.parquet vs OKX-Books-BTC-USDT-400-1744009469406-1744109915504.parquet: 0 items overlap
OKX-Books-BTC-USDT-400-1744009469406-1744109915504.parquet vs OKX-Books-BTC-USDT-400-1744109915604-1744210254701.parquet: 0 items overlap
OKX-Books-BTC-USDT-400-1744109915604-1744210254701.parquet vs OKX-Books-BTC-USDT-400-1744210254801-1744310813203.parquet: 0 items overlap
OKX-Books-BTC-USDT-400-17442102548

In [5]:
total_tss = []
for pf in pfs:
    total_tss.extend(tss[pf])

import matplotlib.pyplot as plt
import numpy as np

# Assuming total_tss is already populated as a list of timestamps (in milliseconds)
# Calculate differences between consecutive timestamps
total_tss.sort()  # Ensure timestamps are sorted
time_diffs = [total_tss[i+1] - total_tss[i] for i in range(len(total_tss)-1)]

# Define time bins (in milliseconds) for 1s, 10s, 1min, 10min, 1h
bins = [0, 1000, 10000, 60000, 600000, 3600000]  # Start from 0
bin_labels = ['0-1s', '1s-10s', '10s-1min', '1min-10min', '10min-1h']

# Count occurrences in each bin
hist, bin_edges = np.histogram(time_diffs, bins=bins, density=False)

# Plotting the histogram with logarithmic y-axis
plt.figure(figsize=(10, 6))
plt.bar(range(len(bin_labels)), hist, tick_label=bin_labels, align='center')
plt.yscale('log')  # Set y-axis to logarithmic scale
plt.xlabel('Time Difference Intervals')
plt.ylabel('Number of Occurrences (Log Scale)')
plt.title('Histogram of Time Differences Between Consecutive Timestamps (Log Scale)')
plt.grid(True, alpha=0.3, which='both')  # Include grid for both major and minor ticks

# Save the plot
plt.savefig('timestamp_diff_histogram_log.png')
plt.close()

In [11]:
stat = {str(i): 0 for i in bins}
stat['inf'] = 0
for diff in time_diffs:
    for i in range(len(bins) - 1):
        if bins[i] <= diff < bins[i + 1]:
            stat[str(bins[i])] += 1
            break
    else:
        stat['inf'] += 1

# to csv file
import pandas as pd
stat_df = pd.DataFrame(list(stat.items()), columns=['Time Interval (ms)', 'Count'])
stat_df.to_csv('timestamp_diff_statistics.csv', index=False)